# Data Manipulation with Pandas
## Data Mining Project — Student Performance Dataset

**Goal:** Practice cleaning, transforming, sorting, grouping, and creating columns in a dataset using Pandas.

**Before you start:** Put your CSV dataset in the same folder as this notebook. The default filename below is the dataset used in this project; change it if your filename is different.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)


## 2. Load the dataset and create a working copy
`df` keeps the loaded data, while `data` is the DataFrame we manipulate.


In [ ]:
FILE_PATH = "b119bb30-d943-4808-ab85-f269b231ebfb.csv"

df = pd.read_csv(FILE_PATH)
data = df.copy()

print("Original shape:", df.shape)
display(data.head())


## 3. Inspect the data
Check column names, data types, missing values, and duplicate rows before changing the data.


In [ ]:
print("Columns:", data.columns.tolist())
print("\\nData types:")
display(data.dtypes.to_frame("Data type"))
print("\\nMissing values per column:")
display(data.isna().sum().to_frame("Missing values"))
print("\\nDuplicate rows:", data.duplicated().sum())


## 4. Standardize column names
This converts column names to lowercase and replaces spaces or punctuation with underscores, making them easier to use in code.


In [ ]:
import re

data.columns = [
    re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z]+", "_", str(col).strip().lower())).strip("_")
    for col in data.columns
]
print(data.columns.tolist())
display(data.head())


## 5. Remove duplicate rows
`drop_duplicates()` removes repeated rows. We keep the first occurrence.


In [ ]:
before = len(data)
data = data.drop_duplicates().copy()
print("Duplicate rows removed:", before - len(data))
print("Shape after removing duplicates:", data.shape)


## 6. Handle missing values
For numeric columns, fill missing values with the column median. For categorical/text columns, fill missing values with the most frequent value when available. This is an example strategy; for a real analysis, choose a method appropriate to the meaning of each column.


In [ ]:
numeric_cols = data.select_dtypes(include="number").columns
categorical_cols = data.select_dtypes(include=["object", "category", "string"]).columns

for col in numeric_cols:
    if data[col].isna().any():
        data[col] = data[col].fillna(data[col].median())

for col in categorical_cols:
    if data[col].isna().any():
        mode_values = data[col].mode(dropna=True)
        if not mode_values.empty:
            data[col] = data[col].fillna(mode_values.iloc[0])

print("Missing values remaining:")
display(data.isna().sum().to_frame("Missing values"))


## 7. Convert data types where appropriate
CSV files sometimes load numbers as text. This cell attempts numeric conversion only for object columns where at least 90% of non-missing values can be interpreted as numbers.


In [ ]:
for col in data.select_dtypes(include=["object", "string"]).columns:
    converted = pd.to_numeric(data[col], errors="coerce")
    non_missing_count = data[col].notna().sum()
    if non_missing_count > 0 and converted.notna().sum() / non_missing_count >= 0.90:
        data[col] = converted

display(data.dtypes.to_frame("Data type after conversion"))


## 8. Create a derived column
This creates `average_score` when the dataset has numeric columns whose names contain `math`, `reading`, and `writing` (a common student-performance schema). If those columns do not exist, the cell prints a message instead of failing.


In [ ]:
score_cols = []
for keyword in ["math", "reading", "writing"]:
    matches = [col for col in data.columns if keyword in col.lower()]
    if matches and pd.api.types.is_numeric_dtype(data[matches[0]]):
        score_cols.append(matches[0])

if len(score_cols) == 3:
    data["average_score"] = data[score_cols].mean(axis=1).round(2)
    print("Created average_score from:", score_cols)
    display(data[score_cols + ["average_score"]].head())
else:
    print("No math/reading/writing score trio was detected.")
    print("Numeric columns available:", data.select_dtypes(include="number").columns.tolist())


## 9. Create a category from a numeric column
For a student score dataset, create a performance label from `average_score`. If that column is unavailable, use the first numeric column as an example. These thresholds are illustrative and can be changed for your task.


In [ ]:
if "average_score" in data.columns:
    target_col = "average_score"
else:
    available_numeric = data.select_dtypes(include="number").columns.tolist()
    target_col = available_numeric[0] if available_numeric else None

if target_col:
    data["performance_level"] = pd.cut(
        data[target_col],
        bins=[-np.inf, 49, 69, 84, np.inf],
        labels=["Needs Improvement", "Satisfactory", "Good", "Excellent"],
        right=True
    )
    print("Created performance_level using:", target_col)
    display(data[[target_col, "performance_level"]].head(10))
else:
    print("No numeric column is available for creating a category.")


## 10. Sort the data
Sort by the average score when available; otherwise, sort by the first numeric column. `ascending=False` places the largest values first.


In [ ]:
numeric_cols_now = data.select_dtypes(include="number").columns.tolist()
sort_col = "average_score" if "average_score" in data.columns else (
    numeric_cols_now[0] if numeric_cols_now else None
)

if sort_col:
    sorted_data = data.sort_values(by=sort_col, ascending=False)
    print("Sorted by:", sort_col)
    display(sorted_data.head(10))
else:
    print("No numeric column is available for sorting.")


## 11. Group data and calculate summaries
For each categorical column, show the mean of numeric columns grouped by category. The example uses the first categorical column with up to 10 groups.


In [ ]:
categorical_cols_now = data.select_dtypes(exclude="number").columns.tolist()
numeric_cols_now = data.select_dtypes(include="number").columns.tolist()

if categorical_cols_now and numeric_cols_now:
    group_col = categorical_cols_now[0]
    summary = data.groupby(group_col, dropna=False)[numeric_cols_now].mean(numeric_only=True).round(2)
    print("Grouped by:", group_col)
    display(summary.head(10))
else:
    print("Grouping summary requires at least one categorical and one numeric column.")


## 12. Rename a column (optional example)
Rename a column by providing a mapping. The cell safely renames the first column as an example; you can edit the mapping to use a meaningful name.


In [ ]:
if len(data.columns) > 0:
    old_name = data.columns[0]
    new_name = "renamed_" + str(old_name)
    data = data.rename(columns={old_name: new_name})
    print(f"Renamed {old_name!r} to {new_name!r}")
display(data.head())


## 13. Export the manipulated dataset
Save the cleaned/transformed result as a new CSV so the original file remains unchanged.


In [ ]:
OUTPUT_PATH = "student_performance_manipulated.csv"
data.to_csv(OUTPUT_PATH, index=False)
print(f"Saved manipulated dataset to: {OUTPUT_PATH}")


## 14. Final checks
Review the final shape, data types, missing values, and a preview of the transformed dataset.


In [ ]:
print("Final shape:", data.shape)
print("Remaining duplicate rows:", data.duplicated().sum())
display(data.isna().sum().to_frame("Missing values"))
display(data.head())


## Summary: Pandas manipulation methods

| Task | Pandas method/example |
|---|---|
| Copy data | `df.copy()` |
| Rename columns | `df.rename(columns={...})` |
| Standardize names | String processing with `re` |
| Remove duplicates | `df.drop_duplicates()` |
| Fill missing values | `fillna()` |
| Convert types | `pd.to_numeric()` |
| Create a new column | `df["new_column"] = ...` |
| Categorize numeric values | `pd.cut()` |
| Sort rows | `df.sort_values()` |
| Group and aggregate | `df.groupby(...).mean()` |
| Export to CSV | `df.to_csv(..., index=False)` |

**Important:** Data manipulation changes or transforms values/structure. This notebook works on a copy and exports a new CSV file, leaving the original dataset untouched.
